# SQuAD Quantized BERT Runner

Run the SQuAD question-answering quantization experiment with `run_squad_quant.py` and `quant_config_squad.json`.


In [ ]:
!pip install transformers==4.35.2
!pip install datasets evaluate fsspec


In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys
from pathlib import Path

os.environ["HF_DATASETS_OFFLINE"] = "0"

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}
!ls {PROJECT_DIR_PATH}


## Config

In [ ]:
CONFIG_PATH = PROJECT_DIR / "quant_config_squad.json"
print("CONFIG_PATH:", CONFIG_PATH)

## Imports

In [ ]:
import torch
from transformers import AutoTokenizer, BertConfig, BertForQuestionAnswering

from mrcp_quant import (
    CustomBertForQuestionAnswering,
    apply_experiment_config,
    apply_layer_quant_overrides,
    load_experiment_config,
    resolve_q_module_list,
    save_experiment_result,
)
from run_squad_quant import calibrate_model, evaluate_model, optimize_scale_factors, q_module_names, quantized_module_paths

## Model Initialization

In [ ]:
experiment_config = load_experiment_config(CONFIG_PATH)
apply_experiment_config(experiment_config)

model_name = experiment_config.get(
    "model_name",
    "bert-large-uncased-whole-word-masking-finetuned-squad",
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("model_name", model_name)
print("device", device)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
hf_config = BertConfig.from_pretrained(model_name)
hf_model = BertForQuestionAnswering.from_pretrained(model_name)

model = CustomBertForQuestionAnswering(hf_config)
applied_layer_quant_overrides = apply_layer_quant_overrides(model, experiment_config)
if applied_layer_quant_overrides:
    print("Applied layer quantization overrides:", applied_layer_quant_overrides)

res = model.load_state_dict(hf_model.state_dict(), strict=False)
print("Missing keys:", len(res.missing_keys))
print("Unexpected keys:", len(res.unexpected_keys))
print("Missing examples:", res.missing_keys[:30])

model.to(device)

## Quantization Setup

In [ ]:
q_module_list = resolve_q_module_list(
    experiment_config.get("q_module_list", ["QLayerNorm"])
)

model.set_q_module_list(q_module_list)
model.set_quant()

note = []
for name, module in model.named_modules():
    q = getattr(module, "quant", None)
    opt = getattr(module, "is_opt_scale", None)
    if (q is True) or (opt is True):
        note.append((name, type(module).__name__, q, opt))

print("modules not in pure-float mode:", len(note))
print(*note[:50], sep="\n")

## Calibration And Scale Optimization

In [ ]:
calibrate_model(model, tokenizer, q_module_list, experiment_config, device)
optimize_scale_factors(model, tokenizer, q_module_list, experiment_config, device)

## Evaluation

In [ ]:
metrics, num_examples = evaluate_model(
    model,
    tokenizer,
    experiment_config,
    device,
)

primary_metric_name = "f1"
primary_metric_value = metrics[primary_metric_name]
print("metrics", metrics)
print("Final exact_match:", metrics["exact_match"])
print("Final f1:", primary_metric_value)

## Save Result

In [ ]:
resolved_q_module_names = q_module_names(q_module_list)
output_config = dict(experiment_config)
output_config["q_module_list"] = resolved_q_module_names

result_path = save_experiment_result(
    accuracy=primary_metric_value,
    loss=None,
    configuration=output_config,
    quantized=resolved_q_module_names,
    output_dir=PROJECT_DIR / "output",
    extra={
        "task_name": "squad",
        "model_name": model_name,
        "quantized_module_paths": quantized_module_paths(model),
        "metrics": metrics,
        "primary_metric_name": primary_metric_name,
        "primary_metric_value": primary_metric_value,
        "num_val_examples": num_examples,
    },
)
print("Saved results:", result_path)